# Machine Learning on a Live Stream

You have sent messages through Kafka and queried them. Now the real thing: **train a model
on past data, then use it to make predictions on data as it arrives.**

This is how fraud detection, spam filtering, predictive maintenance and recommendation
systems actually run in production.

## The problem

You work for a bank. Every card transaction arrives as a message. Some are fraudulent. You
want to flag the suspicious ones **within seconds**, not in tomorrow's report.

## The plan

```
  PART A: learn from the past              PART B: judge the present
  ---------------------------              -------------------------
  5,000 old transactions                   new transaction arrives
  with known outcomes                              |
          |                                        v
          v                                   Kafka topic
   train a model  ----------------------->  model scores it
   (find the pattern)                             |
                                                  v
                                           FRAUD or ok, in seconds
```

The important idea: **the model is trained once, then applied many times.** Training is
slow and happens occasionally. Scoring is fast and happens constantly.

## Before you start

Do `kafka_streaming_lab_basics.ipynb` first. This notebook assumes you know what a topic,
a producer and a consumer are.

Setup takes 2 to 3 minutes. The whole notebook takes about 8.

## Step 1. Setup

The same four install cells as the other notebooks. Run them and wait.

In [ ]:
# --- Java ----------------------------------------------------------------
# Spark 4 needs Java 17 or newer. Colab sometimes already has it, so check before
# installing: the apt install is the slowest part of this cell when it is needed.
import os, subprocess, time, re

t0 = time.time()

def java_major(home=None):
    exe = (home + '/bin/java') if home else 'java'
    try:
        out = subprocess.run([exe, '-version'], capture_output=True, text=True).stderr
    except FileNotFoundError:
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0

JAVA_HOME = '/usr/lib/jvm/java-17-openjdk-amd64'

if java_major() >= 17:
    print('Java 17+ already here, nothing to install')
    JAVA_HOME = subprocess.run(
        ['bash', '-lc', 'dirname $(dirname $(readlink -f $(which java)))'],
        capture_output=True, text=True).stdout.strip()
elif java_major(JAVA_HOME) >= 17:
    print('Java 17 already installed at', JAVA_HOME)
else:
    print('installing Java 17 (JRE only, which is all Spark and Kafka need)...')
    # jre-headless, not jdk-headless: no compiler, so a smaller download.
    !apt-get install -y openjdk-17-jre-headless -qq > /dev/null 2>&1

os.environ['JAVA_HOME'] = JAVA_HOME
os.environ['PATH'] = JAVA_HOME + '/bin:' + os.environ['PATH']
print('JAVA_HOME =', JAVA_HOME)
!java -version
print('--- %.1f seconds ---' % (time.time() - t0))

In [ ]:
# --- PySpark and a Kafka client ------------------------------------------
# kafka-python is a producer you drive from Python. Version 1 sent messages by
# launching kafka-console-producer.sh, which starts a whole JVM for every call:
# about 1 to 2 seconds per message. This sends 8 messages in about 0.15 seconds.
import time
t0 = time.time()

!pip install -q pyspark==4.0.1 kafka-python

import pyspark
print('pyspark     ', pyspark.__version__)
import kafka
print('kafka-python', kafka.__version__)
print('--- %.1f seconds ---' % (time.time() - t0))

In [ ]:
# --- Kafka ----------------------------------------------------------------
# The CDN at dlcdn.apache.org only carries current releases, and which release is
# current changes over the year. So ask it what it has rather than hard-coding a
# version that will quietly disappear. archive.apache.org has every version ever
# released but is throttled to about 120 KB/s, so it is the fallback, not the default.
import os, re, time, urllib.request

KAFKA_DIR = '/content/kafka'
t0 = time.time()

def newest_on_cdn():
    html = urllib.request.urlopen(
        'https://dlcdn.apache.org/kafka/', timeout=30).read().decode()
    vers = sorted({v.rstrip('/') for v in re.findall(r'\d+\.\d+\.\d+/', html)},
                  key=lambda s: [int(p) for p in s.split('.')])
    return vers[-1] if vers else None

if not os.path.exists(KAFKA_DIR + '/bin'):
    try:
        KAFKA_VERSION = newest_on_cdn()
        URL = 'https://dlcdn.apache.org/kafka/%s/kafka_2.13-%s.tgz' % (
            KAFKA_VERSION, KAFKA_VERSION)
        print('CDN has Kafka %s, downloading at CDN speed...' % KAFKA_VERSION)
    except Exception as e:
        KAFKA_VERSION = '3.9.1'
        URL = ('https://archive.apache.org/dist/kafka/%s/kafka_2.13-%s.tgz'
               % (KAFKA_VERSION, KAFKA_VERSION))
        print('CDN unreachable (%s). Falling back to the slow archive.' % e)

    !wget -q --show-progress -O /content/kafka.tgz {URL}
    !mkdir -p {KAFKA_DIR}
    !tar xzf /content/kafka.tgz -C {KAFKA_DIR} --strip-components=1
    !rm -f /content/kafka.tgz
else:
    print('Kafka already unpacked')

!{KAFKA_DIR}/bin/kafka-topics.sh --version
print('--- %.1f seconds ---' % (time.time() - t0))

In [ ]:
# --- The Spark Kafka connector -------------------------------------------
# Spark cannot read Kafka without four extra jars. The usual way is
# .config('spark.jars.packages', ...), which makes Spark run Ivy to resolve them
# from Maven on every start: often a few minutes in Colab, and it sometimes stalls.
# Downloading the four files straight into Spark's own jars folder takes about a
# second, and then the SparkSession starts with no network calls at all.
import os, time, pyspark
t0 = time.time()

JARS = os.path.join(os.path.dirname(pyspark.__file__), 'jars')
MAVEN = 'https://repo1.maven.org/maven2'
SV = pyspark.__version__

# kafka-clients and commons-pool2 versions come from the connector's own pom for
# Spark 4.0.1. If you change the PySpark version, check that pom again.
WANTED = [
    ('%s/org/apache/spark/spark-sql-kafka-0-10_2.13/%s/'
     'spark-sql-kafka-0-10_2.13-%s.jar' % (MAVEN, SV, SV),
     'spark-sql-kafka-0-10_2.13-%s.jar' % SV),
    ('%s/org/apache/spark/spark-token-provider-kafka-0-10_2.13/%s/'
     'spark-token-provider-kafka-0-10_2.13-%s.jar' % (MAVEN, SV, SV),
     'spark-token-provider-kafka-0-10_2.13-%s.jar' % SV),
    ('%s/org/apache/kafka/kafka-clients/3.9.1/kafka-clients-3.9.1.jar' % MAVEN,
     'kafka-clients-3.9.1.jar'),
    ('%s/org/apache/commons/commons-pool2/2.12.0/commons-pool2-2.12.0.jar' % MAVEN,
     'commons-pool2-2.12.0.jar'),
]

for url, name in WANTED:
    dest = os.path.join(JARS, name)
    if not os.path.exists(dest):
        !wget -q -O {dest} {url}
    print('%7.1f KB  %s' % (os.path.getsize(dest) / 1024, name))

print('--- %.1f seconds ---' % (time.time() - t0))

## Step 2. Build the historical data

A model learns from examples. We need transactions where we **already know** whether each
one was fraud, because that is what the model learns the pattern from. This is called
**supervised learning**: every training row carries its answer, called the **label**.

Real banks use years of real transactions. We will generate 5,000 realistic fake ones, so
the notebook is self contained.

### The four clues

| Feature | Why it might indicate fraud |
|---|---|
| `amount` | Thieves tend to spend big before the card is blocked |
| `distance` | How far from the cardholder's home, in km. A purchase 300 km away is odd |
| `is_night` | 1 if between 11pm and 5am. Thieves prefer the small hours |
| `foreign` | 1 if the transaction is in another country |

### The thing that makes this realistic

None of those clues is decisive on its own, and that is deliberate.

Plenty of honest people buy things at 2am, travel abroad, and make large purchases. In the
data below, **18 percent of legitimate transactions happen at night**, and so do **48
percent of fraudulent ones**. Night is a hint, not proof.

If we had made all fraud nocturnal and all honest spending daytime, the model would learn
"night equals fraud", score almost perfectly, and teach you nothing. Easy data makes
useless models. We also flip 6 percent of the labels, because real records contain
mistakes.

In [ ]:
import random

NIGHT = [23, 0, 1, 2, 3, 4, 5]        # 11pm to 5am
DAY = list(range(6, 23))              # 6am to 10pm

rng = random.Random(11)               # fixed seed, so your numbers match the text


def make_transaction():
    """Create one fake transaction, with the honest answer attached."""
    fraud = rng.random() < 0.22                    # 22 percent of rows are fraud

    if fraud:
        hour = rng.choice(NIGHT) if rng.random() < 0.55 else rng.choice(DAY)
        amount = round(max(1.0, rng.gauss(260, 200)), 2)
        distance = round(abs(rng.gauss(120, 110)), 1)
        foreign = 1 if rng.random() < 0.45 else 0
    else:
        # honest cardholders are also out at night, and also travel
        hour = rng.choice(NIGHT) if rng.random() < 0.18 else rng.choice(DAY)
        amount = round(max(1.0, rng.gauss(120, 110)), 2)
        distance = round(abs(rng.gauss(45, 60)), 1)
        foreign = 1 if rng.random() < 0.15 else 0

    label = int(fraud)
    if rng.random() < 0.06:            # 6 percent of records are mislabelled
        label = 1 - label

    return {'amount': amount,
            'distance': distance,
            'hour': hour,
            'is_night': 1 if hour in NIGHT else 0,
            'foreign': foreign,
            'is_fraud': label}


history = [make_transaction() for _ in range(5000)]

print('built %d historical transactions' % len(history))
print('fraud rate: %.1f%%' % (100 * sum(r['is_fraud'] for r in history) / len(history)))
print()
print('three examples:')
for r in history[:3]:
    print('  ', r)

You should see 5,000 transactions with a fraud rate near 24 percent.

**That rate is unrealistically high on purpose.** Real card fraud is well under 1 percent.
At that rate a model that says "never fraud" is 99 percent accurate and completely useless,
which is a genuine and painful problem in the industry. We use 22 percent so the lesson is
about streaming rather than about rare events.

## Step 3. Look at the data before modeling

Never train on data you have not looked at. Two minutes here saves hours later.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName('FraudML')
         .config('spark.sql.shuffle.partitions', '4')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

history_df = spark.createDataFrame(history)
history_df.show(5)

print('How do the two groups differ, on average?')
(history_df.groupBy('is_fraud')
           .agg(F.round(F.avg('amount'), 1).alias('avg_amount'),
                F.round(F.avg('distance'), 1).alias('avg_distance'),
                F.round(F.avg('is_night'), 3).alias('night_rate'),
                F.round(F.avg('foreign'), 3).alias('foreign_rate'),
                F.count('*').alias('rows'))
           .orderBy('is_fraud')
           .show())

Read that table carefully, because it tells you what the model has to work with.

Fraud rows have a higher average amount, a greater distance, and are more often at night
and abroad. **But look at how much the two rows overlap.** Night rate 0.48 against 0.19
means nearly one in five honest transactions is also nocturnal.

That overlap is why this is a real prediction problem and not a lookup table.

## Step 4. Train the model

### Two ideas first

**1. Split the data.** We hold back 20 percent of the rows and do not let the model see
them during training. Then we test on those. Judging a model on rows it memorized tells you
nothing, exactly like marking an exam with the answers written on it.

**2. Spark wants features in one column.** Most ML libraries take a table of columns. Spark
wants all the input numbers packed into a single column called `features`. The tool that
does that packing is `VectorAssembler`, and it is the main thing that looks strange the
first time.

### The pipeline

A **Pipeline** chains steps so they always run in the same order, in training and later in
production. That matters more than it sounds: if you assembled features one way for
training and another way for scoring, your predictions would be quietly wrong. The pipeline
makes that mistake impossible.

Ours has two stages:

```
  VectorAssembler          LogisticRegression
  amount, distance,  --->  [features]  --->  learns the weights  --->  prediction
  is_night, foreign          column
```

**Logistic regression** gives each clue a weight, adds them up, and converts the total into
a probability between 0 and 1. It is the simplest useful classifier, and its weights can be
read and argued about, which matters when a regulator asks why a card was blocked.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

FEATURES = ['amount', 'distance', 'is_night', 'foreign']

# Hold back 20 percent that the model will never see during training.
train_data, test_data = history_df.randomSplit([0.8, 0.2], seed=42)
print('training rows: %d' % train_data.count())
print('testing rows : %d   (held back)' % test_data.count())

pipeline = Pipeline(stages=[
    VectorAssembler(inputCols=FEATURES, outputCol='features'),
    LogisticRegression(featuresCol='features', labelCol='is_fraud'),
])

print()
print('training...')
model = pipeline.fit(train_data)        # this is the learning step
print('done')

`model` now holds everything learned. It is the thing you would save to a file and load
into your production system. Training happened once; scoring can now happen millions of
times.

## Step 5. Is the model any good?

Always ask this before trusting anything. We score the 20 percent the model never saw.

### Three ways to measure, and why accuracy alone lies

| Measure | Question it answers |
|---|---|
| **Accuracy** | What fraction of all transactions did it label correctly? |
| **AUC** | How well does it rank fraud above non-fraud? 0.5 is coin flipping, 1.0 is perfect. |
| **Confusion matrix** | Of the fraud, how much did it catch, and how many honest customers did it accuse? |

Accuracy is the one everybody quotes and the one that misleads most often, as the next cell
shows.

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

predictions = model.transform(test_data)

auc = BinaryClassificationEvaluator(
    labelCol='is_fraud', metricName='areaUnderROC').evaluate(predictions)
correct = predictions.filter(F.col('prediction') == F.col('is_fraud')).count()
accuracy = correct / predictions.count()

print('AUC      : %.4f' % auc)
print('accuracy : %.4f' % accuracy)
print()
print('Confusion matrix:')
(predictions.groupBy('is_fraud', 'prediction')
            .count()
            .orderBy('is_fraud', 'prediction')
            .show())

print('What the model learned (the weight of each clue):')
lr = model.stages[-1]
for name, weight in zip(FEATURES, lr.coefficients):
    print('   %-10s %+.4f' % (name, weight))
print('   %-10s %+.4f' % ('intercept', lr.intercept))

### Reading your results

You should see **AUC around 0.83** and **accuracy around 0.86**.

**Your exact numbers will not match mine.** Spark splits the data according to how it
happened to partition it, which depends on the machine. Anything from about 0.80 to 0.85
AUC is the same result. Chasing the third decimal place of a number that moves between runs
is a good way to waste an afternoon.

**The weights make sense.** `is_night` and `foreign` carry the most weight, around +1.0
each. `amount` and `distance` have small weights only because they are measured in big
units: a weight of +0.006 per dollar still adds up to +1.2 across a 200 dollar difference.
Never judge importance by weight size alone when the features are on different scales.

### Now the uncomfortable part

Look at the confusion matrix. In the run used to write this, it was:

| | predicted ok | predicted fraud |
|---|---|---|
| **actually ok** | 748 | 29 |
| **actually fraud** | **112** | 132 |

Accuracy is 86 percent, which sounds respectable. But of 244 real frauds the model caught
only 132. **It missed 46 percent of the fraud, nearly half.**

Meanwhile it was hardly ever wrong about honest customers: 29 false alarms out of 777. The
model is cautious, and caution against a rare-ish event looks like accuracy while quietly
failing at the job.

This is the single most important lesson in applied classification: a number that sounds
good can hide a model that fails at its actual job. The bank does not care about overall
accuracy. It cares how much fraud it caught, and how many customers it wrongly accused.
Step 7 fixes this.

## Step 6. Why we needed all four clues

A fair question: could one clue have done the job alone? Let us measure it instead of
guessing.

In [ ]:
print('%-12s %s' % ('feature', 'AUC on its own'))
print('-' * 30)
for f in FEATURES:
    solo = Pipeline(stages=[
        VectorAssembler(inputCols=[f], outputCol='features'),
        LogisticRegression(featuresCol='features', labelCol='is_fraud'),
    ]).fit(train_data)
    solo_auc = BinaryClassificationEvaluator(
        labelCol='is_fraud', metricName='areaUnderROC').evaluate(
            solo.transform(test_data))
    print('%-12s %.4f' % (f, solo_auc))

print('-' * 30)
print('%-12s %.4f' % ('all four', auc))

Each clue on its own lands somewhere around 0.65 to 0.72. Together they reach about 0.84.

**That gap is the entire value of the model.** No single rule like "block everything over
300 dollars" would work: it would miss most fraud and annoy a lot of honest customers. The
model's job is to weigh weak, overlapping signals together, which is precisely the thing a
human writing if-statements does badly.

## Step 7. Choosing when to raise an alarm

By default the model says "fraud" when it thinks the probability is above **0.5**. That is
just a default, and nothing says 0.5 is right for your business.

Lower the bar and you catch more fraud but accuse more innocent people. Raise it and the
reverse. There is no setting that wins both, so somebody has to choose, and it is a
business decision rather than a technical one.

Let us measure the trade instead of arguing about it.

In [ ]:
from pyspark.ml.functions import vector_to_array

# 'probability' holds [P(ok), P(fraud)]. Pull out the second number.
scored = predictions.withColumn('p_fraud', vector_to_array('probability')[1]).cache()

total_fraud = scored.filter(F.col('is_fraud') == 1).count()

print('%-11s %8s %8s %11s %8s' % ('threshold', 'caught', 'missed', 'false alarm', 'recall'))
print('-' * 50)
for t in [0.5, 0.4, 0.3, 0.25, 0.2, 0.15]:
    flagged = F.col('p_fraud') >= t
    caught = scored.filter(flagged & (F.col('is_fraud') == 1)).count()
    missed = total_fraud - caught
    false_alarm = scored.filter(flagged & (F.col('is_fraud') == 0)).count()
    print('%-11.2f %8d %8d %11d %7.1f%%'
          % (t, caught, missed, false_alarm, 100 * caught / max(1, total_fraud)))

### What this table is telling you

In the run used to write this:

| threshold | caught | missed | false alarms | recall |
|---|---|---|---|---|
| 0.50 | 132 | 112 | 29 | 54.1% |
| 0.40 | 147 | 97 | 50 | 60.2% |
| 0.30 | 171 | 73 | 113 | 70.1% |
| 0.20 | 192 | 52 | 216 | 78.7% |
| 0.15 | 202 | 42 | 314 | 82.8% |

Going from 0.50 to 0.15 lifts the fraud caught from 54 percent to 83 percent. It also
multiplies false alarms by more than ten, from 29 to 314. Every row of that table is a
real choice somebody has to make.

Which is right depends on costs the model cannot know. If a missed fraud costs 500 dollars
and a false alarm costs a two minute phone call, go low. If a false alarm means blocking a
customer's card on holiday, be careful.

**We will use 0.30 for the rest of this notebook**, catching about 70 percent of fraud at a
moderate false alarm rate. Notice that this is a decision somebody made, not something the
model told us.

## Step 8. Start Kafka and create the topic

Part A is finished: we have a trained model. Part B begins, where transactions arrive live.

In [ ]:
import subprocess, time, os, glob

LOG_DIR = '/content/kraft-logs'

# Where the config lives moved between Kafka 3 and Kafka 4, and so did the format
# command. Detect which layout we got, so this cell keeps working whichever version
# the CDN happens to be serving.
if os.path.exists(KAFKA_DIR + '/config/kraft/server.properties'):
    CONFIG = KAFKA_DIR + '/config/kraft/server.properties'   # Kafka 3.x
    FORMAT_EXTRA = []
else:
    CONFIG = KAFKA_DIR + '/config/server.properties'         # Kafka 4.x
    FORMAT_EXTRA = ['--standalone']   # 4.x refuses to format without this

print('using config:', CONFIG)

def kafka(*args):
    """Run a Kafka command line tool and return its output."""
    cmd = [KAFKA_DIR + '/bin/' + args[0]] + list(args[1:])
    return subprocess.run(cmd, capture_output=True, text=True).stdout

if not os.path.exists(LOG_DIR + '/meta.properties'):
    cluster_id = subprocess.run([KAFKA_DIR + '/bin/kafka-storage.sh', 'random-uuid'],
                                capture_output=True, text=True).stdout.strip()
    print('cluster id:', cluster_id)
    !sed -i "s#^log.dirs=.*#log.dirs={LOG_DIR}#" {CONFIG}
    subprocess.run([KAFKA_DIR + '/bin/kafka-storage.sh', 'format',
                    '-t', cluster_id, '-c', CONFIG,
                    '--ignore-formatted'] + FORMAT_EXTRA,
                   capture_output=True, text=True)
    print('storage formatted at', LOG_DIR)
else:
    print('storage already formatted')

In [ ]:
# Start the broker in the background and wait until it answers.
t0 = time.time()
broker = subprocess.Popen(
    [KAFKA_DIR + '/bin/kafka-server-start.sh', CONFIG],
    stdout=open('/content/broker.log', 'w'), stderr=subprocess.STDOUT)

BOOTSTRAP = 'localhost:9092'

for attempt in range(90):
    if 'Error' not in kafka('kafka-topics.sh', '--bootstrap-server', BOOTSTRAP, '--list'):
        print('broker is up after about %.0f seconds' % (time.time() - t0))
        break
    time.sleep(1)
else:
    print('broker did not start. Read the log with:  !tail -40 /content/broker.log')

In [ ]:
print(kafka('kafka-topics.sh', '--bootstrap-server', BOOTSTRAP,
            '--create', '--topic', 'transactions',
            '--partitions', '3', '--replication-factor', '1'))

## Step 9. Send live transactions

These are **new** transactions. In a real bank they would come from payment terminals. The
key is the card number, so all activity for one card lands in the same partition.

Notice we send `is_fraud` along with them. A real feed would not have it, because nobody
knows yet. We include it only so you can check the model's answers at the end. **The model
is never shown it.**

In [ ]:
import json
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    key_serializer=lambda k: k.encode('utf-8'),
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))


def send_transactions(n):
    """Send n brand new transactions to Kafka."""
    for i in range(n):
        txn = make_transaction()
        card = 'card_%04d' % rng.randint(1, 9999)
        producer.send('transactions', key=card, value=txn)
    producer.flush()
    return n


sent = send_transactions(15)
print('sent %d live transactions' % sent)

## Step 10. Point the model at the stream

Here is the payoff, and it is shorter than you expect.

### The one line that matters

```python
scored = model.transform(stream)
```

That is the **same `model.transform()`** you used on the test data in Step 5. The only
difference is that `stream` is a live Kafka feed rather than a fixed table.

Spark handles it. The model does not know or care whether its input is finished or still
arriving. **A trained model is just a function, and a stream is just rows that keep
coming.** Everything else below is plumbing: reading JSON, and printing results nicely.

In [ ]:
from pyspark.sql.types import (StructType, StructField, DoubleType, IntegerType)

# Describe the shape of our JSON so Spark can parse it.
schema = StructType([
    StructField('amount',   DoubleType()),
    StructField('distance', DoubleType()),
    StructField('hour',     IntegerType()),
    StructField('is_night', IntegerType()),
    StructField('foreign',  IntegerType()),
    StructField('is_fraud', IntegerType()),      # the truth, for checking only
])

stream = (spark.readStream.format('kafka')
          .option('kafka.bootstrap.servers', BOOTSTRAP)
          .option('subscribe', 'transactions')
          .option('startingOffsets', 'earliest')
          .load()
          .selectExpr("CAST(key AS STRING) AS card", "CAST(value AS STRING) AS js")
          .withColumn('t', F.from_json('js', schema))
          .select('card', 't.amount', 't.distance', 't.hour',
                  't.is_night', 't.foreign', 't.is_fraud'))

# The payoff: the trained model applied to a live stream.
scored_stream = model.transform(stream)

print('is the scored output still a stream?', scored_stream.isStreaming)

`True` means Spark will keep scoring new rows as they arrive, forever, without you asking
again.

Now send the results somewhere. `foreachBatch` hands us each small group of new rows as an
ordinary DataFrame, and we print them.

In [ ]:
ALERT_THRESHOLD = 0.30          # the business decision from Step 7

results = scored_stream.withColumn('p_fraud', vector_to_array('probability')[1])

stats = {'seen': 0, 'alerts': 0, 'correct_alerts': 0}


def judge(df, batch_id):
    rows = df.collect()
    if not rows:
        return
    print('--- batch %d: %d new transactions ---' % (batch_id, len(rows)))
    for r in rows:
        stats['seen'] += 1
        alert = r['p_fraud'] >= ALERT_THRESHOLD
        if alert:
            stats['alerts'] += 1
            if r['is_fraud'] == 1:
                stats['correct_alerts'] += 1
        verdict = 'ALERT' if alert else 'ok   '
        truth = 'fraud' if r['is_fraud'] == 1 else 'legit'
        print('   %s  p=%.2f  $%7.2f %5.0fkm night=%d foreign=%d   (really: %s)'
              % (verdict, r['p_fraud'], r['amount'], r['distance'],
                 r['is_night'], r['foreign'], truth))


query = (results.writeStream
         .outputMode('append')
         .foreachBatch(judge)
         .trigger(processingTime='2 seconds')
         .start())

import time
time.sleep(8)
print('scored so far:', stats['seen'])

Each line shows what the model decided and, in brackets, what the truth actually was.

Look for the disagreements. You will see some large night-time foreign purchases that were
genuinely honest, and some small ordinary-looking ones that were fraud. That is not a bug.
It is what an 0.82 AUC looks like from close up: usually right, regularly wrong, far better
than guessing.

## Step 11. Keep the transactions coming

The query above is still running. Send more and watch them get scored as they land, with no
further instruction from you.

In [ ]:
print('sending 20 more transactions, a few at a time...')
for wave in range(4):
    send_transactions(5)
    time.sleep(3)

time.sleep(4)
query.stop()

print()
print('=== final tally ===')
print('transactions scored : %d' % stats['seen'])
print('alerts raised       : %d' % stats['alerts'])
if stats['alerts']:
    print('alerts that were real fraud: %d  (%.0f%% of alerts were right)'
          % (stats['correct_alerts'],
             100 * stats['correct_alerts'] / stats['alerts']))

## Step 12. Stop everything

Spark first, then the producer, then Kafka last.

In [ ]:
for q in spark.streams.active:
    q.stop()

producer.close()
spark.stop()
broker.terminate()
broker.wait(timeout=30)
print('everything stopped')

---

# What you just did

```
  5,000 past transactions  ->  pipeline.fit()  ->  model
                                                     |
  live Kafka transactions  ->  model.transform()  ->  ALERT or ok
```

## The five things to remember

1. **Train once, score many times.** Training took seconds and happens occasionally. Scoring
   runs forever. They are separate jobs with completely different costs.

2. **`model.transform()` does not care whether its input is a table or a stream.** This is
   the whole trick. A model is a function; a stream is rows that keep coming.

3. **A Pipeline keeps training and scoring identical.** Features were assembled the same way
   in both, because the same pipeline object did it. Doing that by hand twice is a classic
   source of silently wrong predictions.

4. **Accuracy can hide a weak model.** 86 percent accuracy sounded fine while the model was
   missing nearly half the fraud.

5. **The threshold is a business decision, not a technical one.** The model gives you a
   probability. What to do at 0.31 is a question about the cost of a missed fraud against
   the cost of an angry customer, and no amount of modeling answers it.

## What a real system adds

| We did | Production does |
|---|---|
| Kept the model in memory | Saves it with `model.save()`, loads it in the scoring job |
| Printed alerts | Writes them to a database, or to another Kafka topic |
| Trained once, here | Retrains weekly, because fraud patterns change |
| Used 4 clues | Uses hundreds, including the card's own recent history |
| Trusted the model | Monitors it, and alerts when its scores start drifting |

The one worth dwelling on is **retraining**. A fraud model trained on last year's patterns
slowly stops working as criminals change tactics. This is called **model drift**, and it is
the reason ML systems need maintenance rather than just deployment.

## Glossary

| Term | Meaning |
|---|---|
| **Supervised learning** | Learning from examples where the answer is known. |
| **Label** | The known answer in training data. Ours was `is_fraud`. |
| **Feature** | One input clue the model uses. |
| **VectorAssembler** | Packs several columns into the single `features` column Spark wants. |
| **Pipeline** | A fixed chain of steps, so training and scoring always match. |
| **Logistic regression** | A classifier that weighs each clue and returns a probability. |
| **Train/test split** | Holding back data so you can judge the model honestly. |
| **AUC** | How well the model ranks fraud above non-fraud. 0.5 is useless, 1.0 is perfect. |
| **Confusion matrix** | The four way table of right and wrong, by class. |
| **Recall** | Of all the real fraud, what fraction did we catch? |
| **False alarm** | An honest transaction flagged as fraud. |
| **Threshold** | The probability above which you act. Default 0.5, rarely the right choice. |
| **Model drift** | A model getting worse over time as the world changes. |

## If something went wrong

| What you saw | What to do |
|---|---|
| `NameError: name 'model' is not defined` | Run Step 4 before Step 10. |
| `broker did not start` | `!tail -40 /content/broker.log`, then re-run from Step 1. |
| Nothing prints in Step 10 | Check you ran Step 9, and that `producer.flush()` was called. |
| Numbers differ from the text | Expected, and fine. Spark's split depends on partitioning, so AUC anywhere near 0.80 to 0.85 is the same result. |
| `Failed to find data source: kafka` | Setup cell 4 did not finish. Re-run it, restart the session, start again. |

## Try these

1. Change `ALERT_THRESHOLD` to 0.15 and re-run Steps 10 and 11. More alerts, and more wrong
   ones. Decide which you would ship.
2. Drop `foreign` from `FEATURES` and retrain. How much AUC do you lose?
3. Swap `LogisticRegression` for `RandomForestClassifier` (import it from
   `pyspark.ml.classification`). Does AUC improve enough to justify a model you can no longer
   read the weights of?
4. Make the problem realistic: change the fraud rate from 0.22 to 0.01 and retrain. Watch
   accuracy rise above 99 percent while the model becomes useless. This is the single most
   important experiment on this page.